# ATP MATCHES

## Imports

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as f
from pyspark.sql.window import Window

import pandas as pd

import os
os.environ['SPARK_LOCAL_IP'] = '127.0.0.1'

from dotenv import load_dotenv
load_dotenv()

True

## Init spark

In [2]:
try:
    spark = SparkSession.builder.appName("silver_atp_matches").getOrCreate()
except Exception as e:
    print(e)

In [3]:
spark.conf.set("spark.sql.repl.eagerEval.enabled", True)

spark.conf.set("spark.sql.repl.eagerEval.maxNumRows", 200)
spark.conf.set("spark.sql.repl.eagerEval.truncate", 50)

## Load database

In [4]:
tb_atp_matches = (
    spark.read
    .format("jdbc")
    .option("url", os.getenv("JDBC_URL"))
    .option("dbtable", "bronze.tb_atp_matches")
    .option("user", os.getenv("DB_USER"))
    .option("password", os.getenv("DB_PASSWORD"))
    .option("driver", "org.postgresql.Driver")
    .load()
    )

In [5]:
tb_atp_tournaments = (
    spark.read
    .format("jdbc")
    .option("url", os.getenv("JDBC_URL"))
    .option("dbtable", "silver.tb_atp_tournaments")
    .option("user", os.getenv("DB_USER"))
    .option("password", os.getenv("DB_PASSWORD"))
    .option("driver", "org.postgresql.Driver")
    .load()
    ) 

In [6]:
tb_atp_players = (
    spark.read
    .format("jdbc")
    .option("url", os.getenv("JDBC_URL"))
    .option("dbtable", "silver.tb_atp_players")
    .option("user", os.getenv("DB_USER"))
    .option("password", os.getenv("DB_PASSWORD"))
    .option("driver", "org.postgresql.Driver")
    .load()
    ) 

## ATP MATCHES DATAFRAME

In [5]:
from pyspark.sql import functions as f
from pyspark.sql.window import Window

matches_cleaned = (
    tb_atp_matches
    .withColumn(
        "loser_id",
        f.when(
            f.col("loser_id").isNull() & f.col("loser_name").contains("Alejandro Davidovich Fokina"), '200221'
        ).when(
            f.col("loser_id").isNull() & f.col("loser_name").contains("Shevchenko"), '207686'
        ).when(
            f.col("loser_id").isNull() & f.col("loser_name").contains("Jesper De Jong"), '207411'
        ).otherwise(f.col("loser_id"))
    )
    .withColumn(
        'tourney_id',
        f.when(
            (f.col("tourney_id") == '2026-416') & (f.col('tourney_name') == 'Munich'),
            f.lit("2026-308")
        ).otherwise(f.col("tourney_id"))
    )
)

window_spec = Window.partitionBy("tourney_id").orderBy(
    f.to_date(f.col("tourney_date").cast("string"), "yyyyMMdd").asc(),
    f.col("match_num").asc_nulls_last()
)

df = (
    matches_cleaned
    .withColumn(
        "match_num",
        f.coalesce(
            f.col("match_num").cast("int"),
            f.row_number().over(window_spec)
        )
    )
    .alias('m')

    .join(
        tb_atp_tournaments.alias('t'), 
        (f.col('m.tourney_id') == f.col('t.TOURNEY_ID')),
        'left'
    )
    .join(
        tb_atp_players.alias('p_w'),
        f.col("p_w.PLAYER_ID") == f.col("m.winner_id"),
        'left'
    )
    .join(
        tb_atp_players.alias('p_l'),
        f.col("p_l.PLAYER_ID") == f.col("m.loser_id"),
        'left'
    )

    .withColumn("MATCH_ID", f.concat_ws('-', f.col("t.TOURNEY_ID"), f.col("m.match_num")))

    .select(
        # match / tourney columns
        f.col("MATCH_ID"),
        f.col("t.TOURNEY_ID").alias("TOURNEY_ID"),
        f.col("m.draw_size").alias("MATCH_DRAW_SIZE"),
        f.col("m.tourney_date").alias("MATCH_DATE"),
        f.col("m.score").alias("MATCH_SCORE"),
        f.col("m.best_of").alias("MATCH_BEST_OF"),
        f.col("m.round").alias("MATCH_ROUND"),
        f.col("m.minutes").alias("MATCH_DURATION_M"),

        # winner columns
        f.col("p_w.PLAYER_ID").alias("PLAYER_W_ID"),
        f.col("m.winner_rank").alias("PLAYER_W_RANK"),
        f.col("m.winner_rank_points").alias("PLAYER_W_RANK_PTS"),
        f.col("m.winner_seed").alias("PLAYER_W_SEED"),
        f.upper(f.col("m.winner_entry")).alias("PLAYER_W_ENTRY"),
        f.col("m.w_ace").alias("PLAYER_W_ACES"),
        f.col("m.w_df").alias("PLAYER_W_DB_FAULTS"),
        f.col("m.w_svpt").alias("PLAYER_W_SERVE_PTS"),
        f.col("m.w_1stIn").alias("PLAYER_W_1ST_SERVES_IN"),
        f.col("m.w_1stWon").alias("PLAYER_W_1ST_SERVE_PTS_WON"),
        f.col("m.w_2ndWon").alias("PLAYER_W_2ND_SERVE_PTS_WON"),
        f.col("m.w_SvGms").alias("PLAYER_W_SERVE_GAMES"),
        f.col("m.w_bpSaved").alias("PLAYER_W_BP_SAVED"),
        f.col("m.w_bpFaced").alias("PLAYER_W_BP_FACED"),
        
        # loser columns
        f.col("p_l.PLAYER_ID").alias("PLAYER_L_ID"),
        f.col("m.loser_rank").alias("PLAYER_L_RANK"),
        f.col("m.loser_rank_points").alias("PLAYER_L_RANK_PTS"),
        f.col("m.loser_seed").alias("PLAYER_L_SEED"),
        f.upper(f.col("m.loser_entry")).alias("PLAYER_L_ENTRY"),
        f.col("m.l_ace").alias("PLAYER_L_ACES"),
        f.col("m.l_df").alias("PLAYER_L_DB_FAULTS"),
        f.col("m.l_svpt").alias("PLAYER_L_SERVE_PTS"),
        f.col("m.l_1stIn").alias("PLAYER_L_1ST_SERVES_IN"),
        f.col("m.l_1stWon").alias("PLAYER_L_1ST_SERVE_PTS_WON"),
        f.col("m.l_2ndWon").alias("PLAYER_L_2ND_SERVE_PTS_WON"),
        f.col("m.l_SvGms").alias("PLAYER_L_SERVE_GAMES"),
        f.col("m.l_bpSaved").alias("PLAYER_L_BP_SAVED"),
        f.col("m.l_bpFaced").alias("PLAYER_L_BP_FACED")
    )
)

## Validate

In [6]:
if tb_atp_matches.count() == df.count():
    print('ok')
else:
    raise

ok


## Save dataframe

### Local

In [7]:
df.toPandas().to_csv(
    r"../../data/silver/tb_atp_matches.csv",
    index=False,
    sep=",",
    encoding="utf-8"
)

### Supabase

In [56]:
(
df.write
    .format("jdbc")
    .option("url", os.getenv("JDBC_URL"))
    .option("dbtable", "silver.tb_atp_matches")
    .option("user", os.getenv("DB_USER"))
    .option("password", os.getenv("DB_PASSWORD"))
    .option("driver", "org.postgresql.Driver")
    .mode("overwrite")
    .save()
)